# CIDS · 00 — Generate narrative (Type-N) prompts

**Settings:** Accelerator = GPU T4 x2 (or P100), Internet = ON.  
**Inputs:** attach the `cids-assets` dataset (contains `cids_prompts.csv`).  
If the repo is private, add a Kaggle Secret named `GITHUB_TOKEN` (Add-ons → Secrets).

Run 1 (all 1506 cases, Qwen2.5-7B) → Run 2 (306 test cases, Phi-3.5-mini, held-out wording). Both runs are resumable: if the session stops, just run the notebook again.

In [ ]:
REPO = 'https://github.com/Vemuri-BK/cids.git'   # <- change if your repo name differs
BRANCH = 'main'
WORK = '/kaggle/working'
OUT = f'{WORK}/narratives'

In [ ]:
import os, subprocess
token = ''
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
except Exception:
    pass
url = REPO.replace('https://', f'https://{token}@') if token else REPO
if os.path.isdir(f'{WORK}/cids/.git'):
    subprocess.run(['git', '-C', f'{WORK}/cids', 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', '-b', BRANCH, url, f'{WORK}/cids'], check=True)
%cd /kaggle/working/cids
!git log -1 --oneline

In [ ]:
!pip install -q -U bitsandbytes sentence-transformers accelerate
!pytest -q tests

In [ ]:
from pathlib import Path
cands = list(Path('/kaggle/input').rglob('cids_prompts.csv'))
assert len(cands) == 1, cands
PROMPTS = str(cands[0]); print(PROMPTS)

## Smoke test (16 cases) — check the printed pass rates before the full run

In [ ]:
!python scripts/generate_narratives.py --prompts {PROMPTS} --target train_style --out_dir {OUT}/smoke --limit 16

In [ ]:
import json
recs = [json.loads(l) for l in open(f'{OUT}/smoke/train_style_progress.jsonl')]
for r in recs[:3]:
    print(r['patient_id'], r['status'])
    for t in r['chosen']: print('   ', t)
    bad = [c for c in r['candidates'] if c['problems']]
    if bad: print('   rejected example:', bad[0]['problems'])

## Run 1 — train-style narratives, all cases (~1–3 h on T4)

In [ ]:
!python scripts/generate_narratives.py --prompts {PROMPTS} --target train_style --out_dir {OUT}

## Run 2 — held-out narratives from a different LLM, test cases only

In [ ]:
!python scripts/generate_narratives.py --prompts {OUT}/cids_prompts_with_train_style.csv --target heldout --out_dir {OUT}

In [ ]:
import shutil
shutil.copy(f'{OUT}/cids_prompts_with_heldout.csv', f'{WORK}/cids_prompts_final.csv')
import pandas as pd
d = pd.read_csv(f'{WORK}/cids_prompts_final.csv', keep_default_na=False)
print({c: int((d[c].str.len()>0).sum()) for c in d.columns if c.startswith('narrative')})
# Download cids_prompts_final.csv from the Output panel, then locally:
# python scripts/update_excel.py --csv cids_prompts_final.csv --xlsx data/prompts/cids_prompts.xlsx